# NOE (Hilbert space and Redfield)
## Author: Vineeth Francis Thalakottoor, vineeth.thalakottoor@cea.fr
- Nuclear Overhauser effect 
- In Hilbert Space
- Redfield Master equation
- Heteronuclear

In [ ]:
# Define the source path
SourcePath = "/home/vineeth/Documents/PyOR/PyOR"

# Add source path
import sys
sys.path.append(SourcePath)

import time
%matplotlib ipympl

# Import PyOR package
from PyORv2 import *

In [ ]:
# Define the spin system
Spin_list = {"A" : "C13", "B" : "H1"}
QS = QunS(Spin_list,PrintDefault=False)

### Set parameters

In [ ]:
QS.Configure(
    # Propagation Space
    PropagationSpace="Hilbert",

    # Master Equation
    MasterEquation="Redfield",

    # Operator basis
    Basis_SpinOperators_Hilbert="Zeeman",

    # Field in TFalse
    B0=9.4,

    # Offset frequencies in Hz
    OFFSET={"A": 10.0, "B": 50.0},

    # J coupling
    Jcouplings=[("A", "B", 5.0)],

    # Define paris of spins coupled by dipolar interaction
    Dipole_Pairs = [("A","B")],

    # Initial and final spin temperature
    I_spintemp={"A": 300.0, "B": 300.0},
    F_spintemp={"A": 300.0, "B": 300.0},
    Lindblad_Temp = 300.0,

    # Relaxation Process
    Rprocess = "Auto-correlated Dipolar Heteronuclear",
    RelaxParDipole_tau = 10.0e-12,
    RelaxParDipole_bIS = [30.0e3],

    # Evolution parameters
    AcqDT = 0.0001,
    AcqAQ = 30.0,
    OdeMethod = 'DOP853',
    PropagationMethod = "ODE Solver",
    
    #Plotting
    PlotFigureSize = (10,5),
    PlotFontSize = 20,
)

### Generate Hamiltonians

In [ ]:
Hz = QS.Hamiltonian.Zeeman_RotFrame()
Hz.Inverse2PI().Round(3).matrix

In [ ]:
# J coupling Hamiltonian
Hj = QS.Hamiltonian.Jcoupling()
Hj.Inverse2PI().Round(3).matrix

### product operator basis (Shift Z or PMZ basis)

In [ ]:
sort = 'negative to positive'
Index = False
Normal = True
Basis_PMZ, coh_PMZ, dic_PMZ = QS.Basis.ProductOperators_SpinHalf_PMZ(sort,Index,Normal)

### Initialize Density Matrix

In [ ]:
Thermal_DensMatrix = True

if Thermal_DensMatrix:    
    # High Temperature
    HT_approx = False
    
    # Initial Density Matrix
    rho_in = QS.DensityMatrix.EquilibriumDensityMatrix(QS.Ispintemp,HT_approx)
    
    # Equlibrium Density Matrix
    rhoeq = QS.DensityMatrix.EquilibriumDensityMatrix(QS.Fspintemp,HT_approx)
else:
    rho_in = QS.Az + QS.Bz
    rhoeq = QS.Az + QS.Bz

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho_in,Basis_PMZ,dic_PMZ)

### Pulse

In [ ]:
flip_angle1 = 0.0   # Flip angle Spin 1
flip_angle2 = 180.0 # Flip angle Spin 2

rho = QS.HardPulse.Rotate_Pulse(rho_in,flip_angle1,QS.Ay)
rho = QS.HardPulse.Rotate_Pulse(rho,flip_angle2,QS.By) 

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho,Basis_PMZ,dic_PMZ)

## Evolution 

In [ ]:
start_time = time.time()
t, rho_t = QS.Evolutions.Evolution(rho,rhoeq,Hz+Hj)
end_time = time.time()
timetaken = end_time - start_time
print("Total time = %s seconds " % (timetaken))

### Expectation Value

In [ ]:
det_Z1 = QS.Az
det_Z2 = QS.Bz

t, signal_Z1 = QS.Evolutions.Expectation(rho_t,det_Z1)
t, signal_Z2 = QS.Evolutions.Expectation(rho_t,det_Z2)

In [ ]:
MoA = rho_in.Expectation(QS.Az)
MoB = rho_in.Expectation(QS.Bz)

## Plotting

In [ ]:
QS.Plotting.PlottingMulti([t,t],[signal_Z1/MoA,signal_Z2/MoB],"time (s)","Mz",["green","blue"])